# 06 — Consolidação dos Entregáveis Executivos, KPIs Finais e Gráficos

**Narrativa Executiva Oficial:**
> “O portfólio aloca R$ 11,12M de custo bruto para gerar R$ 26,08M de GMV incremental, 352,7 mil pedidos incrementais líquidos e R$ 2,10M de margem de contribuição incremental. Após custos operacionais e de SLA, o impacto líquido na Operating Margin é exatamente -R$ 10M, conforme a restrição do case. O retorno é de R$ 0,19 de MC incremental por R$ 1 de incentivo e 2,35x de múltiplo de GMV.”


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

PROJECT_DIR = "/content/drive/MyDrive/ifood-growth-competition-case"
if os.path.exists("/content/drive"):
    if not os.path.exists(PROJECT_DIR):
        from google.colab import drive
        drive.mount("/content/drive")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        PROJECT_DIR = os.getcwd()

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

for sub in ["data/raw", "data/processed", "outputs/excel", "outputs/figures", "outputs/reports", "dashboard"]:
    os.makedirs(sub, exist_ok=True)

print("Diretório ativo:", os.getcwd())


In [ ]:
df_base = pd.read_csv("data/processed/base_validada.csv")
df_alloc = pd.read_csv("data/processed/base_alocacao_otimizada.csv")
df_sens = pd.read_csv("outputs/reports/sensibilidade_cenarios.csv")

inv_bruto = df_alloc["custo_bruto_rs"].sum()
ped_liq = df_alloc["pedidos_incrementais_liquidos"].sum()
gmv_inc = df_alloc["gmv_incremental_rs"].sum()
mc_inc = df_alloc["mc_incremental_rs"].sum()
custo_op = df_alloc["custo_operacional_sla_rs"].sum()
om_inc = df_alloc["impacto_liquido_om_rs"].sum()
mc_por_real = mc_inc / inv_bruto
mult_gmv = gmv_inc / inv_bruto

gmv_base_tot = df_base["gmv"].sum()
om_base_tot = df_base["operating_margin_rs"].sum()
om_pos_tot = om_base_tot + om_inc
om_base_pct = om_base_tot / gmv_base_tot * 100.0
om_pos_pct = om_pos_tot / (gmv_base_tot + gmv_inc) * 100.0

summary_kpis = pd.DataFrame([
    ("Cidades ativas", f"{len(df_alloc)} ({', '.join(df_alloc['cidade'])})"),
    ("Investimento bruto total", f"R$ {inv_bruto:,.2f}"),
    ("Pedidos incrementais líquidos", f"{ped_liq:,.2f}"),
    ("GMV incremental", f"R$ {gmv_inc:,.2f}"),
    ("MC incremental", f"R$ {mc_inc:,.2f}"),
    ("Custo operacional/SLA", f"R$ {custo_op:,.2f}"),
    ("Impacto líquido na Operating Margin", f"R$ {om_inc:,.2f}"),
    ("MC incremental por R$ 1 de custo bruto", f"R$ {mc_por_real:.4f}"),
    ("Múltiplo de GMV", f"{mult_gmv:.2f}x"),
    ("Operating Margin consolidada", f"R$ {om_base_tot/1e6:.2f}M -> R$ {om_pos_tot/1e6:.2f}M; {om_base_pct:.2f}% -> {om_pos_pct:.2f}%"),
], columns=["Indicador", "Valor Final Aprovado"])

display(summary_kpis)
summary_kpis.to_csv("outputs/reports/kpis_finais_consolidados.csv", index=False)


In [ ]:
# Painel Executivo Consolidado (2x2) para Apresentação e Dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. GMV Incremental por Cidade
b1 = axes[0, 0].bar(df_alloc["cidade"], df_alloc["gmv_incremental_rs"]/1e6, color=["#EA1D2C", "#B91C1C", "#F59E0B", "#DC2626", "#1E293B"], width=0.52)
axes[0, 0].set_title("GMV Incremental por Cidade (R$ Milhões)", fontweight="bold")
axes[0, 0].set_ylabel("R$ Milhões")
axes[0, 0].set_ylim(0, 9.0)
for b in b1:
    axes[0, 0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.18, f"R$ {b.get_height():.2f}M", ha="center", fontweight="bold", fontsize=9)

# 2. Impacto Líquido na Operating Margin por Cidade
b2 = axes[0, 1].bar(df_alloc["cidade"], df_alloc["impacto_liquido_om_rs"]/1e6, color=["#334155"]*4 + ["#991B1B"], width=0.52)
axes[0, 1].set_title("Impacto Líquido na Operating Margin (Soma = -R$ 10,00M)", fontweight="bold")
axes[0, 1].set_ylabel("R$ Milhões")
axes[0, 1].set_ylim(-3.2, 0)
for b in b2:
    axes[0, 1].text(b.get_x() + b.get_width()/2, b.get_height() - 0.25, f"R$ {b.get_height():.2f}M", ha="center", fontweight="bold", fontsize=9)

# 3. City 17 — Saturação
b3 = axes[1, 0].bar(["Base Real", "Sem Intervenção", "Pós-Investimento"], [131.38, 135.88, 121.54], color=["#64748B", "#DC2626", "#15803D"], width=0.45)
axes[1, 0].axhline(100, color="#0F172A", linestyle=":", linewidth=1.1)
axes[1, 0].set_title("City 17: Saturação (%) — Alívio de -14,35 p.p. vs. Sem Intervenção", fontweight="bold")
axes[1, 0].set_ylim(0, 155)
for b in b3:
    axes[1, 0].text(b.get_x() + b.get_width()/2, b.get_height() + 3, f"{b.get_height():.2f}%", ha="center", fontweight="bold", fontsize=9)

# 4. City 6 — Guardrail 98%
b4 = axes[1, 1].bar(["Base Real", "Pós-Investimento", "Teto Guardrail"], [96.50, 97.54, 98.00], color=["#64748B", "#F59E0B", "#EA1D2C"], width=0.45)
axes[1, 1].set_title("City 6: Guardrail de Saturação (Folga = 0,46 p.p.)", fontweight="bold")
axes[1, 1].set_ylim(90, 100)
for b in b4:
    axes[1, 1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.22, f"{b.get_height():.2f}%", ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/figures/06_painel_executivo_consolidado.png", dpi=180)
plt.show()
